# Figure 8 EXP Ward profiles

This is the original radar-chart code for panels a–d. The spatial map in panel e is not included.

Input files are referenced by name; place the existing inputs in `INPUT_DIR` or point it to their location. Only figure files are saved if the plotting cells are run.


In [ ]:
from pathlib import Path
import pandas as pd

# Point this to the existing input files for the selected panel.
INPUT_DIR = Path(".")
OUTPUT_DIR = Path("figure_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## Panels a–d Four standardised EXP profiles

Source: `CES_EXP_Aggregation_and_Ward_Clustering.ipynb`, cells 84, 90, 92 and 94. The original Ward clustering and radar styling use the June 26 evidence-filtered percentages. The park-code lookup is from cell 45.


In [ ]:
import math
park_name_to_code_map = {
    "香山-植物园": "HG1",
    "圆明园": "HG2",
    "颐和园": "HG3",
    "八大处公园": "P1",
    "巴沟山水园": "P2",
    "百望山森林公园": "P3",
    "北京西山国家森林公园": "P4",
    "北坞公园": "P5",
    "畅春新园体育休闲广场": "P6",
    "茶棚公园": "P7",
    "国家植物园": "P8",
    "船营公园": "P9",
    "丹青圃公园": "P10",
    "东升八家郊野公园": "P11",
    "功德寺公园": "P12",
    "海淀公园": "P13",
    "荷清园": "P14",
    "六郎庄公园": "P15",
    "妙云寺公园": "P16",
    "南旱河公园": "P17",
    "上地公园": "P18",
    "石渠公园": "P19",
    "树村公园": "P20",
    "王庄公园": "P21",
    "万柳公园": "P22",
    "肖家河公园": "P23",
    "西水磨公园": "P24",
    "一亩园公园": "P25",
    "影湖楼公园": "P26",
    "玉东郊野公园": "P27",
    "中坞公园": "P28",
}

import os
import csv
import zipfile
import numpy as np
import matplotlib.pyplot as plt
import glob

from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
from matplotlib import font_manager, rcParams

import pandas as pd
from datetime import datetime
import seaborn as sns # Import seaborn for color palettes

print("--- Starting Ward Hierarchical Clustering for Experience Sentiment Profiles ---")

# ------------------------------------------------------------
# 1. File path and output folder
# ------------------------------------------------------------

output_dir = "experience_Ward_clustering_4class_outputs"
# Create output directory relative to current working directory or specific path
output_base_path = str(OUTPUT_DIR)
full_output_dir = os.path.join(output_base_path, output_dir)
os.makedirs(full_output_dir, exist_ok=True)

comfort_percentages_df = pd.read_csv(INPUT_DIR / "comfort_sentiment_percentages_table_20260626155035.csv", index_col="scenic_spot")
park_names = comfort_percentages_df.index.tolist()
comments = comfort_percentages_df["Total Comments"].tolist()

# ------------------------------------------------------------
# 3. Define input variables (features for clustering)
# ------------------------------------------------------------
# SNT_1_PCT = positive percentage
# SNT_0_PCT = negative percentage

feature_cols = [
    "Accessibility_SNT_1_PCT",
    "Accessibility_SNT_0_PCT",
    "Safety_SNT_1_PCT",
    "Safety_SNT_0_PCT",
    "Usability_SNT_1_PCT",
    "Usability_SNT_0_PCT",
    "Comfort_SNT_1_PCT",
    "Comfort_SNT_0_PCT"
]

feature_labels = [
    "Accessibility Positive",
    "Accessibility Negative",
    "Safety Positive",
    "Safety Negative",
    "Usability Positive",
    "Usability Negative",
    "Comfort Positive",
    "Comfort Negative"
]

short_labels = [
    "Access\n+",
    "Access\n-",
    "Safety\n+",
    "Safety\n-",
    "Usability\n+",
    "Usability\n-",
    "Comfort\n+",
    "Comfort\n-"
]

# Matrix: rows = parks, columns = 8 experience sentiment indicators
X = comfort_percentages_df[feature_cols].values


# ------------------------------------------------------------
# 4. Z-score standardisation
# ------------------------------------------------------------
# z = (x - mean) / standard deviation

scaler = StandardScaler()
X_zscore = scaler.fit_transform(X)

means = scaler.mean_
stds = scaler.scale_


# ------------------------------------------------------------
# 5. Ward hierarchical clustering
# ------------------------------------------------------------

Z = linkage(
    X_zscore,
    method="ward",
    metric="euclidean"
)

# Set number of clusters
n_clusters = 4
print(f"Performing clustering with {n_clusters} clusters.")

raw_labels = fcluster(
    Z,
    t=n_clusters,
    criterion="maxclust"
)

raw_cluster_ids = sorted(np.unique(raw_labels))


# ------------------------------------------------------------
# 6. Calculate cluster centroids
# ------------------------------------------------------------

raw_centroids = {
    c: X[raw_labels == c].mean(axis=0)
    for c in raw_cluster_ids
}

z_centroids = {
    c: X_zscore[raw_labels == c].mean(axis=0)
    for c in raw_cluster_ids
}

idx = {
    name: i
    for i, name in enumerate(feature_labels)
}


# ------------------------------------------------------------
# 7. Rename clusters as EXP-P1 to EXP-P4 based on characteristics
# ------------------------------------------------------------

# Assign descriptive names based on z-score profiles, similar to CES example
# These assignments are heuristic and can be adjusted based on domain knowledge

# EXP-P1: high positive accessibility and safety
p1_raw = max(
    raw_cluster_ids,
    key=lambda c:
    z_centroids[c][idx["Accessibility Positive"]]
    + z_centroids[c][idx["Safety Positive"]]
)

remaining = [c for c in raw_cluster_ids if c != p1_raw]

# EXP-P2: high usability and comfort positive, low negative experience
p2_raw = max(
    remaining,
    key=lambda c:
    z_centroids[c][idx["Usability Positive"]]
    + z_centroids[c][idx["Comfort Positive"]]
    - z_centroids[c][idx["Accessibility Negative"]]
    - z_centroids[c][idx["Comfort Negative"]]
)

remaining = [c for c in remaining if c != p2_raw]

# EXP-P3: constraint-sensitive profile (high negative accessibility/safety/comfort, low usability positive)
p3_raw = max(
    remaining,
    key=lambda c:
    z_centroids[c][idx["Accessibility Negative"]]
    + z_centroids[c][idx["Safety Negative"]]
    + z_centroids[c][idx["Comfort Negative"]]
    - z_centroids[c][idx["Usability Positive"]]
)

# EXP-P4: remaining mixed moderate type
p4_raw = [c for c in remaining if c != p3_raw][0]


cluster_map = {
    p1_raw: (
        "EXP-P1",
        "高可达性-高安全性公园"
    ),
    p2_raw: (
        "EXP-P2",
        "高可用性-高舒适度公园"
    ),
    p3_raw: (
        "EXP-P3",
        "受限敏感型体验公园"
    ),
    p4_raw: (
        "EXP-P4",
        "混合中等体验公园"
    )
}

cluster_type_english_map = {
    "EXP-P1": "High Accessibility-High Safety Parks",
    "EXP-P2": "High Usability-High Comfort Parks",
    "EXP-P3": "Constraint-sensitive Experience Parks",
    "EXP-P4": "Mixed Moderate Experience Parks"
}

cluster_codes = [
    cluster_map[c][0]
    for c in raw_labels
]

cluster_types = [
    cluster_map[c][1]
    for c in raw_labels
]

cluster_order = [
    "EXP-P1",
    "EXP-P2",
    "EXP-P3",
    "EXP-P4"
]


In [ ]:
# Re-order the raw cluster IDs to match the cluster_order (EXP-P1 to EXP-P4)
ordered_raw_cluster_ids = [
    key for exp_code in cluster_order for key, (code, _) in cluster_map.items() if code == exp_code
]

# Prepare data for plotting (cluster centroids in Z-score values)
cluster_z_means = np.array([z_centroids[c] for c in ordered_raw_cluster_ids])

num_vars = len(feature_labels)
angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
angles += angles[:1]

# Custom color palette for clusters
exp_cluster_colors_map = {
    'EXP-P1': '#7FBD6C', # Greenish
    'EXP-P2': '#A9CCE3', # Bluish
    'EXP-P3': '#EC7063', # Reddish
    'EXP-P4': '#F7DC6F'  # Yellowish
}

# Determine dynamic y-axis limits for scaled data based on all Z-score values
all_z_values = X_zscore.flatten() # All standardized data points
min_z = np.min(all_z_values)
max_z = np.max(all_z_values)

# Ensure 0 is visible and set symmetric limits if appropriate
z_range_abs = max(abs(min_z), abs(max_z))
# Increase range slightly to ensure all data points fit within plot boundaries
y_lower_lim = -z_range_abs * 1.1
y_upper_lim = z_range_abs * 1.1

# Create ticks at integer values within the calculated range
z_ticks_candidates = np.arange(np.floor(y_lower_lim), np.ceil(y_upper_lim) + 1).astype(int)
# Filter out ticks that are too close or outside reasonable range for display clarity
z_ticks = [t for t in z_ticks_candidates if t >= np.floor(min_z*1.2) and t <= np.ceil(max_z*1.2) and t % 1 == 0]

# Add 0 if not present, and ensure at least some ticks exist
if 0 not in z_ticks: z_ticks.append(0)
z_ticks = sorted(list(set(z_ticks)))


In [ ]:
# ------------------------------------------------------------
# 10. Plotting: Panel Radar Charts (Cluster Centroids + Individual Parks)
# ------------------------------------------------------------

print("--- 正在生成面板雷达图 (经验情绪聚类中心及个体公园) ---")

# `cluster_z_means` contains the z-score centroids (from previous step)

# Prepare individual park data (also in Z-score)
scaled_park_df = pd.DataFrame(X_zscore, index=park_names, columns=feature_cols)

num_clusters_plot = len(ordered_raw_cluster_ids)
cols_panel = 2 # 2 columns for panel charts
rows_panel = math.ceil(num_clusters_plot / cols_panel)

fig_panel_radar, axes_panel_radar = plt.subplots(rows_panel, cols_panel, figsize=(16, 8 * rows_panel), subplot_kw=dict(polar=True))
axes_panel_radar = axes_panel_radar.flatten()

individual_park_palette = sns.color_palette('tab20', 20) # A palette with 20 distinct colors

# Max value for y-axis scale based on Z-score for all data points
y_max_limit_z = np.max(np.abs(all_z_values)) * 1.1 # 10% buffer

for i, raw_c_id in enumerate(ordered_raw_cluster_ids):
    ax_p = axes_panel_radar[i]
    exp_code, exp_type_cn = cluster_map[raw_c_id]
    exp_type_en = cluster_type_english_map[exp_code]
    plot_color = exp_cluster_colors_map[exp_code]

    # Plot cluster centroid
    mean_z = cluster_z_means[ordered_raw_cluster_ids.index(raw_c_id)]
    mean_z_values_closed = mean_z.tolist() + mean_z[:1].tolist() # Close the loop
    ax_p.plot(angles, mean_z_values_closed, color=plot_color, linewidth=3, linestyle='solid', label='Cluster Mean')
    ax_p.fill(angles, mean_z_values_closed, color=plot_color, alpha=0.25)

    # Plot individual parks in the cluster
    parks_in_this_cluster_mask = (raw_labels == raw_c_id)
    current_cluster_park_names = [name for k, name in enumerate(park_names) if parks_in_this_cluster_mask[k]]

    legend_handles_panel = [plt.Line2D([0], [0], color=plot_color, lw=3, linestyle='-', label='Cluster Mean')]
    legend_labels_panel = ['Cluster Mean']

    for park_idx, park_name_full in enumerate(current_cluster_park_names):
        park_z_data = scaled_park_df.loc[park_name_full, feature_cols].values
        park_z_data_closed = park_z_data.tolist() + park_z_data[:1].tolist() # Close the loop

        park_color = individual_park_palette[park_idx % len(individual_park_palette)]
        park_code_for_legend = park_name_to_code_map.get(park_name_full, park_name_full) # Use code if available

        ax_p.plot(angles, park_z_data_closed, color=park_color, linewidth=1.5, linestyle='--', alpha=0.8, label=park_code_for_legend)
        legend_handles_panel.append(plt.Line2D([0], [0], color=park_color, lw=1.5, linestyle='--', label=park_code_for_legend))
        legend_labels_panel.append(park_code_for_legend)


    ax_p.set_theta_offset(np.pi / 2)
    ax_p.set_theta_direction(-1)

    ax_p.set_xticks(angles[:-1])
    ax_p.set_xticklabels(short_labels, color='darkgreen', size=9)

    ax_p.set_ylim(y_lower_lim, y_upper_lim) # Use the same global z-score limits as in mean radar
    ax_p.set_yticks(z_ticks)
    ax_p.set_yticklabels([str(t) for t in z_ticks], color='grey', size=8)

    title_text = f"{exp_code}: {exp_type_en}\n{len(current_cluster_park_names)} parks"
    ax_p.set_title(title_text, size=12, color='black', y=1.12, fontweight='bold', pad=25) # Increased pad

    # Adjust legend position to avoid overlapping with title
    ax_p.legend(handles=legend_handles_panel, labels=legend_labels_panel, loc='upper right', bbox_to_anchor=(1.5, 1.1), fontsize=8, frameon=False, ncol=2)

# Hide any unused subplots
for j in range(num_clusters_plot, len(axes_panel_radar)):
    fig_panel_radar.delaxes(axes_panel_radar[j])

fig_panel_radar.tight_layout()
# fig_panel_radar.subplots_adjust(wspace=0.6, hspace=0.7) # Fine-tune spacing

panel_radar_path = os.path.join(full_output_dir, f'experience_zscore_{n_clusters}cluster_panel_radar_{datetime.now().strftime("%Y%m%d%H%M%S")}.png')
fig_panel_radar.savefig(panel_radar_path, dpi=300)
plt.close(fig_panel_radar)

print(f"面板雷达图已保存至: {panel_radar_path}")
